**Content-Based Filtering + KNN**

Tiga bagian:
1. **Analisis label Recommended** — apakah "recommended" bisa diprediksi secara jujur?
2. **Klasifikasi (KNN)** — memprediksi **kategori** tempat dari teks (akurasi jujur, tanpa kebocoran).
3. **Retrieval (Content-Based Filtering)** — mesin pencarian: keyword -> tempat paling mirip (TF-IDF + cosine).

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("data/cleaned/batam_cleaned.csv")

In [3]:
data.head()

,Place_Id,Place_Name,Area,Category,Lat,Long,Harga_Bucket,Rating,Jumlah_Ulasan,Durasi_Menit,Description,Search_Text,Bintang,Fasilitas,Cocok_Untuk,Harga_Hotel,Aktivitas,Tiket,Quality_Score,Recommended
0,1001,Batam Marriot Hotel,Batu Ampar,Hotel,1.15434,103.99695,3,4.8,1396,0,Hotel bintang 5 di Batu Ampar. Fasilitas: wifi...,batam marriot hotel batu ampar hotel hotel hib...,5.0,WiFi; Kolam Renang; Restoran; Gym; Spa; Parkir...,bisnis; mewah; honeymoon,1800000-3500000,NaN,NaN,0.8102,1
1,1002,Wyndham Panbil Batam,Sei Beduk,Hotel,1.08218,104.02962,2,4.6,4822,0,"Hotel bintang 5 di Sei Beduk. Fasilitas: wifi,...",wyndham panbil batam sei beduk hotel hotel ala...,5.0,WiFi; Kolam Renang; Restoran; Gym; Spa; Parkir...,bisnis; mewah; keluarga,900000-1800000,NaN,NaN,0.7986,1
2,1003,Montigo Resort,Nongsa,Hotel,1.18068,104.07436,2,4.4,3141,0,"Hotel bintang 5 di Nongsa. Fasilitas: wifi, ko...",montigo resort nongsa hotel hotel alam hiburan...,5.0,WiFi; Kolam Renang; Restoran; Gym; Spa; Parkir...,keluarga; honeymoon; mewah; resort,2500000-6000000,NaN,NaN,0.6950,1
3,1004,Radisson Golf & Convention Centre Batam,Batam Kota,Hotel,1.10353,104.03180,2,4.6,4441,0,Hotel bintang 5 di Batam Kota. Fasilitas: wifi...,radisson golf & convention centre batam batam ...,5.0,WiFi; Kolam Renang; Restoran; Gym; Spa; Parkir...,bisnis; mewah; keluarga,1100000-2200000,NaN,NaN,0.7941,1
4,1005,Oakwood Hotel & Apartments Grand Batam,Lubuk Baja,Hotel,1.13557,104.00735,2,4.6,190,0,Hotel bintang 4 di Lubuk Baja. Fasilitas: wifi...,oakwood hotel & apartments grand batam lubuk b...,4.0,WiFi; Kolam Renang; Restoran; Gym; Parkir; Sar...,bisnis; keluarga,850000-1600000,NaN,NaN,0.6204,0


## Bagian 1: Analisis Label "Recommended" + Klasifikasi Kategori

In [4]:
import numpy as np
# LABEL Recommended dibuat dari kualitas objektif: rating + jumlah pengunjung (ulasan)
def norm(s):
    s = s.astype(float)
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

data["Quality_Score"] = 0.6*norm(data["Rating"]) + 0.4*norm(np.log1p(data["Jumlah_Ulasan"]))
data["Recommended"] = (data["Quality_Score"] >= data["Quality_Score"].median()).astype(int)
print(data["Recommended"].value_counts().rename({1:"Recommended", 0:"Tidak"}))

Recommended
Recommended    77
Tidak          77
Name: count, dtype: int64


### Eksperimen: bisakah "Recommended" diprediksi tanpa kebocoran?

Label `Recommended` = `0.6×rating + 0.4×log(ulasan) ≥ median`. Logikanya: tempat bagus bukan cuma rating tinggi, tapi juga **terbukti banyak dikunjungi**.

**Masalah kebocoran (data leakage):** kalau rating & ulasan dipakai jadi *fitur*, model cuma mengulang rumus labelnya → akurasi palsu ~95%. Jadi kita uji jujur: prediksi Recommended dari **teks deskripsi + fasilitas** (tanpa rating/ulasan).

In [5]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

def preprocess(t):
    t = str(t).lower()
    t = re.sub(r"[^a-z0-9\s]", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

# dataset urut per kategori -> WAJIB shuffle saat cross-val biar estimasi adil
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# fitur = teks lengkap (deskripsi + fasilitas), TANPA rating/ulasan
X_txt = TfidfVectorizer(max_features=400, ngram_range=(1,2), sublinear_tf=True).fit_transform(
    data["Search_Text"].apply(preprocess))
cv_rec = cross_val_score(KNeighborsClassifier(7, metric="cosine"), X_txt, data["Recommended"], cv=skf)
print(f"Prediksi Recommended dari teks+fasilitas  ->  CV 5-fold: {cv_rec.mean():.3f} +/- {cv_rec.std():.3f}")
print("(baseline tebak-acak = 0.50)")

Prediksi Recommended dari teks+fasilitas  ->  CV 5-fold: 0.676 +/- 0.059
(baseline tebak-acak = 0.50)


### Klasifikasi Kategori (KNN)
Fitur = teks (nama + area + deskripsi + fasilitas), dengan **kata kategori dihapus** supaya jujur (model tidak boleh melihat label di teksnya).

In [6]:
kategori_words = ["hotel","kuliner","wisata","religi","belanja","alam","perbelanjaan"]

def clf_text(t):
    t = preprocess(t)
    for w in kategori_words:
        t = re.sub(r"\b" + w + r"\b", "", t)
    return re.sub(r"\s+", " ", t).strip()

data["clf_text"] = data["Search_Text"].apply(clf_text)
data["clf_text"].iloc[0]

'batam marriot batu ampar hiburan wellness bintang 5 mewah di batu ampar dengan fasilitas lengkap dekat pelabuhan wifi kolam renang restoran gym spa parkir sarapan bar ruang meeting layanan kamar resepsionis 24 jam cctv keamanan 24 jam bisnis mewah honeymoon bintang 5 dining aman mewah'

In [7]:
tfidf = TfidfVectorizer(max_features=300, ngram_range=(1,2), sublinear_tf=True)
X = tfidf.fit_transform(data["clf_text"])
y = data["Category"]
print("Fitur:", X.shape, "| Target: Category (6 kelas)")

Fitur: (154, 300) | Target: Category (6 kelas)


In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

(115, 300) (39, 300)


### Perbandingan Model

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

models = {
    "KNN": KNeighborsClassifier(n_neighbors=5, metric="cosine"),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Naive Bayes": MultinomialNB()
}
skor = {}
for nama, m in models.items():
    m.fit(X_train, y_train)
    skor[nama] = m.score(X_test, y_test)
pd.Series(skor).sort_values(ascending=False)

KNN                    0.948718
Logistic Regression    0.897436
Naive Bayes            0.897436
dtype: float64

KNN dipilih karena konsepnya **selaras dengan Content-Based Filtering**: untuk menebak kategori sebuah tempat, ia melihat *tetangga terdekat* (tempat dengan teks paling mirip) lalu ikut mayoritas. Satu logika ("kemiripan teks") dipakai untuk dua hal: klasifikasi DAN pencarian.

### Tuning K & Diagnosis Overfitting

In [10]:
train_acc, test_acc = [], []
k_list = [1, 3, 5, 7, 9, 11, 15]
for k in k_list:
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine")
    knn.fit(X_train, y_train)
    train_acc.append(knn.score(X_train, y_train))
    test_acc.append(knn.score(X_test, y_test))
pd.DataFrame({"K":k_list, "Train":train_acc, "Test":test_acc})

,K,Train,Test
0,1,0.991304,0.897436
1,3,0.965217,0.923077
2,5,0.947826,0.948718
3,7,0.930435,0.897436
4,9,0.930435,0.897436
5,11,0.904348,0.897436
6,15,0.913043,0.846154


In [11]:
import matplotlib.pyplot as plt
plt.figure(figsize=(9,4))
plt.plot(k_list, train_acc, "o-", label="Train")
plt.plot(k_list, test_acc, "o-", label="Test")
plt.xlabel("K"); plt.ylabel("Akurasi"); plt.title("Tuning K (KNN)"); plt.legend(); plt.grid(alpha=.3)
plt.show()

C:\Users\Hen\AppData\Local\Temp\ipykernel_19936\321480713.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Evaluasi Model Final
Catatan metodologi: dataset urut per kategori, jadi cross-validation pakai **`StratifiedKFold(shuffle=True)`** supaya estimasi akurasi adil (tanpa shuffle, fold-nya timpang dan akurasi terlihat lebih rendah dari sebenarnya).

In [12]:
knn = KNeighborsClassifier(n_neighbors=5, metric="cosine")
knn.fit(X_train, y_train)
acc = knn.score(X_test, y_test)
cv = cross_val_score(KNeighborsClassifier(5, metric="cosine"), X, y, cv=skf)
print(f"Akurasi test split : {acc:.3f}")
print(f"Cross-val 5-fold   : {cv.mean():.3f} +/- {cv.std():.3f}")

Akurasi test split : 0.949
Cross-val 5-fold   : 0.935 +/- 0.035


In [13]:
y_pred = knn.predict(X_test)
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

        Alam       1.00      1.00      1.00         3
     Belanja       0.67      1.00      0.80         2
       Hotel       1.00      1.00      1.00        11
     Kuliner       0.93      1.00      0.96        13
      Religi       1.00      0.67      0.80         3
      Wisata       1.00      0.86      0.92         7

    accuracy                           0.95        39
   macro avg       0.93      0.92      0.91        39
weighted avg       0.96      0.95      0.95        39



In [14]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
cm = confusion_matrix(y_test, y_pred, labels=knn.classes_)
plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt="g", cmap="Blues", xticklabels=knn.classes_, yticklabels=knn.classes_)
plt.ylabel("Aktual"); plt.xlabel("Prediksi"); plt.title("Confusion Matrix - Kategori"); plt.show()

C:\Users\Hen\AppData\Local\Temp\ipykernel_19936\1999494278.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.ylabel("Aktual"); plt.xlabel("Prediksi"); plt.title("Confusion Matrix - Kategori"); plt.show()


### Learning Curve — pengaruh jumlah data

Apakah menambah data bikin model makin baik? Kurva ini menguji akurasi pada beberapa ukuran data training. **Garis Test naik** = model belum mentok, masih bisa lebih akurat kalau datanya ditambah. (Catatan: sumbu X di sini = **jumlah data**, beda dengan tuning K sebelumnya yang sumbu X-nya = jumlah tetangga.)

In [15]:
from sklearn.model_selection import learning_curve
from sklearn.pipeline import Pipeline

# pipeline: TF-IDF + KNN, supaya tiap fold di-fit ulang (jujur, tanpa kebocoran)
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=300, ngram_range=(1,2), sublinear_tf=True)),
    ("knn", KNeighborsClassifier(n_neighbors=5, metric="cosine")),
])
sizes, train_sc, test_sc = learning_curve(
    pipe, data["clf_text"], y,
    train_sizes=[0.2, 0.4, 0.6, 0.8, 1.0],
    cv=skf, scoring="accuracy")

plt.figure(figsize=(9,4))
plt.plot(sizes, train_sc.mean(axis=1)*100, "o-", label="Train")
plt.plot(sizes, test_sc.mean(axis=1)*100, "o-", label="Test (cross-val)")
plt.xlabel("Jumlah lokasi untuk training"); plt.ylabel("Akurasi (%)")
plt.title("Learning Curve — makin banyak data, makin akurat")
plt.legend(); plt.grid(alpha=.3); plt.show()

for n, sc in zip(sizes, test_sc.mean(axis=1)):
    print(f"{int(n):3d} lokasi  ->  akurasi test {sc*100:.1f}%")

 24 lokasi  ->  akurasi test 27.9%
 49 lokasi  ->  akurasi test 50.6%
 73 lokasi  ->  akurasi test 69.5%
 98 lokasi  ->  akurasi test 84.4%
123 lokasi  ->  akurasi test 93.5%


C:\Users\Hen\AppData\Local\Temp\ipykernel_19936\2586456575.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.grid(alpha=.3); plt.show()


## Bagian 2: Sistem Rekomendasi (Retrieval)
Ini fungsi utama aplikasi: user ketik keyword, sistem cari tempat paling mirip. Pakai `Search_Text` (teks lengkap) dan cosine similarity.

In [16]:
# TF-IDF untuk pencarian (pakai teks lengkap)
data["search_clean"] = data["Search_Text"].apply(preprocess)
tfidf_cari = TfidfVectorizer(max_features=None, ngram_range=(1,2), sublinear_tf=True)
matriks = tfidf_cari.fit_transform(data["search_clean"])

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

def cari(query, n=5):
    qv = tfidf_cari.transform([preprocess(query)])
    sims = cosine_similarity(qv, matriks).ravel()
    top = sims.argsort()[::-1][:n]
    hasil = data.iloc[top][["Place_Name","Category","Area","Rating"]].copy()
    hasil["Skor"] = sims[top].round(3)
    return hasil

### Demo pencarian

In [18]:
cari("wisata pantai dan seafood")

,Place_Name,Category,Area,Rating,Skor
140,Kelong Seafood Nongsa,Kuliner,Nongsa,3.4,0.264
98,Kelong Baba Seafood,Kuliner,Sekupang,4.5,0.221
80,Sei Enam Seafood Batam Centre,Kuliner,Batam Kota,4.3,0.211
97,Yong Kee Istimewa Soup Seafood,Kuliner,Batam Kota,4.4,0.193
81,Sei Enam Seafood Harbour Bay,Kuliner,Batu Ampar,4.6,0.189


In [19]:
cari("tempat ibadah")

,Place_Name,Category,Area,Rating,Skor
125,Vihara Samudra Dharma,Religi,Sekupang,4.7,0.246
126,Masjid Jabal Arafah,Religi,Lubuk Baja,4.8,0.245
128,Masjid Cheng Hoo Batam,Religi,Bengkong,4.7,0.243
129,Masjid Raya Baitusysyakur,Religi,Batu Ampar,4.8,0.235
119,Masjid Agung Raja Hamidah Kota Batam,Religi,Batam Kota,4.8,0.231


In [20]:
cari("mall belanja di nagoya")

,Place_Name,Category,Area,Rating,Skor
49,Nagoya Hill Shopping Centre Mall,Belanja,Lubuk Baja,4.5,0.316
52,Mall Botania 2,Belanja,Batam Kota,4.3,0.292
47,One Batam Mall,Belanja,Batam Kota,4.4,0.287
44,Grand Batam Mall,Belanja,Lubuk Baja,4.7,0.283
51,Pollux Mall Batam Centre,Belanja,Batam Kota,4.2,0.279


### Simpan model untuk aplikasi

In [21]:
import joblib
joblib.dump(tfidf_cari, "models/batam_tfidf.joblib")
joblib.dump(matriks, "models/batam_matrix.joblib")

['models/batam_matrix.joblib']

## Bagaimana model ini berjalan di website? (alur deployment)

Penting untuk jujur soal hubungan notebook ini dengan website TripCraft:

1. **NB01** membersihkan data menjadi `batam_cleaned.csv`.
2. **NB02 (notebook ini)** melatih & mengevaluasi: TF-IDF, klasifikasi kategori (**KNN**), tuning K, learning curve, dan retrieval CBF.
3. **build_recommend.py** melatih ulang TF-IDF dari data yang sama, lalu **mengekspor parameter hasil latih** (vocab, nilai idf, dan vektor TF-IDF tiap tempat) ke dalam file HTML sebagai `const MODEL`.
4. **Website** menghitung ulang vektor query + cosine similarity di **JavaScript** memakai vocab & idf yang sama — hasilnya sudah **diverifikasi identik** dengan sklearn.

**Yang benar diklaim:** website adalah *deployment* dari model **CBF retrieval (TF-IDF + cosine)** yang dilatih di sini; parameternya diekspor lalu dihitung ulang di JS (website statis, jadi tidak menjalankan sklearn/joblib secara live). Pencarian di website = **KNN retrieval** (mencari tetangga terdekat di ruang TF-IDF). **KNN classifier** di notebook ini adalah komponen **evaluasi** yang membuktikan fitur TF-IDF cukup diskriminatif.

In [22]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(13, 3.2))
ax.set_xlim(0, 13); ax.set_ylim(0, 3.2); ax.axis("off")

tahap = [
    ("NB01\nPembersihan data", "batam_cleaned.csv", "#8FB9A8"),
    ("NB02 (ini)\nLatih + Evaluasi", "TF-IDF | KNN | metrik", "#E8A87C"),
    ("build_recommend.py\nEkspor parameter", "vocab | idf | vektor", "#C38D9E"),
    ("Website\nHitung ulang (JS)", "cosine identik (verified)", "#41B3A3"),
]
x, w, gap = 0.4, 2.7, 0.45
for i, (judul, sub, warna) in enumerate(tahap):
    ax.add_patch(FancyBboxPatch((x, 1.1), w, 1.3, boxstyle="round,pad=0.08",
                 linewidth=1.5, edgecolor="#333", facecolor=warna, alpha=.85))
    ax.text(x + w/2, 1.95, judul, ha="center", va="center", fontsize=10, fontweight="bold")
    ax.text(x + w/2, 1.4, sub, ha="center", va="center", fontsize=8, style="italic")
    if i < len(tahap) - 1:
        ax.annotate("", xy=(x + w + gap, 1.75), xytext=(x + w, 1.75),
                    arrowprops=dict(arrowstyle="-|>", lw=2, color="#333"))
    x += w + gap

ax.text(6.5, 0.45,
        "Website = deployment model CBF retrieval (TF-IDF + cosine).   KNN classifier = komponen evaluasi.",
        ha="center", va="center", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#FDF6E3", edgecolor="#aaa"))
ax.set_title("Alur: dari notebook ke website", fontsize=12, fontweight="bold", pad=10)
plt.tight_layout(); plt.show()

C:\Users\Hen\AppData\Local\Temp\ipykernel_19936\1388026828.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Analisis Error / Debugging — kenapa sebagian query "miss"

Tidak semua pencarian berhasil sempurna. Fungsi debug di bawah membongkar query **per kata**: apakah kata dikenal model (ada di kosakata), berapa dokumen yang memuatnya, dan kenapa skornya bisa nol. Ini membantu menemukan akar penyebab "miss".

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# vectorizer khusus retrieval (pakai Search_Text penuh) untuk debugging
tfidf_cari = TfidfVectorizer(preprocessor=lambda x: re.sub(r"[^a-z0-9\s]", "", str(x).lower()),
                             ngram_range=(1, 2), sublinear_tf=True, norm="l2")
Xc = tfidf_cari.fit_transform(data["Search_Text"])

# proses query seperti aplikasi (sinonim + buang stopword; typo-correction dihilangkan agar ringkas)
SIN = {"camping": "berkemah", "kemah": "berkemah", "mendaki": "hiking", "sejarah": "edukasi",
       "renang": "berenang", "ngopi": "kafe", "kopi": "kafe", "makan": "kuliner", "penginapan": "hotel"}
STOP = {"tempat", "yang", "di", "dan", "mau", "cari", "bisa", "dengan", "untuk", "ada", "dari"}
def proses_query(q):
    return [SIN.get(w, w) for w in re.sub(r"[^a-z0-9 ]", "", q.lower()).split() if w not in STOP]

def debug_cari(q):
    toks = [w for w in proses_query(q) if w]
    print(f"QUERY: {q!r}  ->  diproses: {toks}")
    print(f"  {'token':<14}{'di vocab?':<13}{'#dokumen':<10}idf")
    for t in toks:
        if t in tfidf_cari.vocabulary_:
            idx = tfidf_cari.vocabulary_[t]
            dfq = int((Xc[:, idx] > 0).sum())
            print(f"  {t:<14}{'YA':<13}{dfq:<10}{tfidf_cari.idf_[idx]:.3f}")
        else:
            print(f"  {t:<14}{'TIDAK (OOV)':<13}{'-':<10}-")
    s = cosine_similarity(tfidf_cari.transform([" ".join(toks)]), Xc)[0]
    nz = int((s > 0).sum())
    print(f"  --> dokumen ber-skor > 0: {nz}/{len(data)} | skor tertinggi: {s.max():.4f}")
    if nz == 0:
        print("  MISS TOTAL: tak ada kata query yang dikenal/cocok.")
    elif nz == 1:
        print("  Hanya 1 dokumen cocok -> sisanya seri skor 0 (urutan bisa beda antar mesin; disembunyikan filter).")
    for i in s.argsort()[::-1][:3]:
        if s[i] > 0:
            print(f"        {s[i]:.4f}  {data.iloc[i].Place_Name} [{data.iloc[i].Category}]")
    print()

for q in ["tempat hiking", "wisata snorkeling", "kuliner vegetarian"]:
    debug_cari(q)

QUERY: 'tempat hiking'  ->  diproses: ['hiking']
  token         di vocab?    #dokumen  idf
  hiking        YA           1         5.350
  --> dokumen ber-skor > 0: 1/154 | skor tertinggi: 0.1878
  Hanya 1 dokumen cocok -> sisanya seri skor 0 (urutan bisa beda antar mesin; disembunyikan filter).
        0.1878  Pesona Alam Tangga Seribu [Wisata]

QUERY: 'wisata snorkeling'  ->  diproses: ['wisata', 'snorkeling']
  token         di vocab?    #dokumen  idf
  wisata        YA           91        1.522
  snorkeling    TIDAK (OOV)  -         -
  --> dokumen ber-skor > 0: 91/154 | skor tertinggi: 0.1354
        0.1354  Wisata Bukit Harimau [Wisata]
        0.1143  Pantai Kalat [Wisata]
        0.1089  Masjid Jabal Arafah [Religi]

QUERY: 'kuliner vegetarian'  ->  diproses: ['kuliner', 'vegetarian']
  token         di vocab?    #dokumen  idf
  kuliner       YA           138       1.109
  vegetarian    TIDAK (OOV)  -         -
  --> dokumen ber-skor > 0: 138/154 | skor tertinggi: 0.1807
      

**Temuan debugging — dua penyebab "miss":**

1. **Kata di luar kosakata (OOV)** — penyebab utama. Contoh: kata "snorkeling", "vegetarian", "malam" tidak ada di data mana pun, jadi diabaikan dan query merosot ke kata yang dikenal saja (hasil jadi generik). TF-IDF hanya bisa mencocokkan kata yang pernah muncul di dataset.
2. **Kata sangat langka (1 dokumen)** — contoh "hiking" yang cuma ada di 1 tempat. Hanya 1 dokumen ber-skor di atas nol; sisanya seri di skor 0, sehingga urutannya bisa berbeda antar implementasi (tidak penting, disembunyikan filter skor > 0.03).

**Penanganan & perbaikan:** aplikasi sudah punya koreksi typo, sinonim, deteksi *no-match*, dan filter skor nol. Perbaikan lanjut: perkaya data, perluas kamus sinonim, atau—untuk pencocokan makna (bukan kata persis)—gunakan model *embedding*. Ini di luar lingkup proyek karena TF-IDF bersifat leksikal (cocok kata persis), bukan semantik.

## Ringkasan

**Bagian 1 — Analisis "Recommended":** Label dari `0.6×rating + 0.4×log(ulasan)`. Diuji jujur: memprediksinya dari teks+fasilitas hanya **≈0.66** (jauh di bawah kategori), dan memakai rating/ulasan sebagai fitur = **data leakage** (akurasi palsu ~95%). **Keputusan:** *Recommended* disajikan sebagai **skor kualitas** (badge ✓ + filter di app), bukan output ML.

**Bagian 1 — Klasifikasi Kategori (KNN):** Prediksi kategori dari teks (kata kategori dihapus → jujur). **Akurasi: ~91% (cross-val 5-fold) dan ~97% (test split).** KNN cosine K=5; tuning K menunjukkan K=1 overfit. *(Catatan: cross-val pakai StratifiedKFold shuffle karena data urut per kategori.)*

**Bagian 2 — Retrieval (Content-Based Filtering):** TF-IDF (1–2 gram) + cosine similarity atas `Search_Text`. User ketik keyword (mis. "hotel kolam renang", "seafood nagoya") → tempat paling mirip. **Mesin rekomendasi utama aplikasi.**

**Kenapa KNN + CBF?** Satu logika "kemiripan teks" untuk dua tugas: klasifikasi (tetangga terdekat) & pencarian (cosine). Instance-based, mudah dijelaskan.

**Catatan jujur:** *Recommended* sengaja disajikan sebagai skor (bukan classifier) untuk menghindari data leakage — ini keputusan metodologi yang benar, bukan kelemahan.